# 00 Setup And Smoke

Install optional local-model dependencies, set the shared Kaggle partition contract, and generate a local-kaggle command manifest without calling Gemini or databases.

This notebook also verifies that the import scripts and the `w3_local_outputs/` layout expected by the later notebooks are available.

In [ ]:
import json, os
from pathlib import Path

CORPUS_SLUG = 'tuvi-golden-corpus'
SCRIPTS_SLUG = 'tuvi-battu-scripts'

ALL_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']
SOURCE_IDS = ['TVKL', 'TVNL', 'TVHS', 'TVGM']
RUN_TAG = 'part_a'
PARTITION_MODE = 'strategy'  # recommended: split Kaggle versions by strategy
SELECTED_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child']
SELECTED_SOURCES = list(SOURCE_IDS)
INPUT_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/dinhbaobao'),
    Path('/kaggle/input'),
]

def resolve_input_root() -> Path:
    for candidate in INPUT_ROOT_CANDIDATES:
        if candidate.exists():
            return candidate
    return Path('/kaggle/input')

def resolve_input_path(value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return INPUT_ROOT / path

INPUT_ROOT = resolve_input_root()
CORPUS_ROOT = resolve_input_path(CORPUS_SLUG)
SCRIPTS_ROOT = resolve_input_path(SCRIPTS_SLUG)
CORPUS_DIR = CORPUS_ROOT / 'benchmark' / 'tuvi_golden_dataset'
SCRIPTS_DIR = SCRIPTS_ROOT
if not CORPUS_DIR.exists():
    CORPUS_DIR = Path.cwd() / 'benchmark' / 'tuvi_golden_dataset'
if not SCRIPTS_DIR.exists():
    SCRIPTS_DIR = Path.cwd()

SOURCE_REGISTRY = CORPUS_DIR / 'guideline' / 'source_registry.json'

if PARTITION_MODE not in {'strategy', 'source'}:
    raise ValueError('PARTITION_MODE must be strategy or source')

ACTIVE_STRATEGIES = [item for item in ALL_STRATEGIES if item in SELECTED_STRATEGIES] if PARTITION_MODE == 'strategy' else list(ALL_STRATEGIES)
ACTIVE_SOURCES = [item for item in SOURCE_IDS if item in SELECTED_SOURCES] if PARTITION_MODE == 'source' else list(SOURCE_IDS)
if not ACTIVE_STRATEGIES:
    raise ValueError('No active strategies selected')
if not ACTIVE_SOURCES:
    raise ValueError('No active sources selected')

OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
REPORTS = OUTPUT_ROOT / 'reports'
STATE = OUTPUT_ROOT / 'state'
PAYLOADS = OUTPUT_ROOT / 'payloads'
EMBEDDINGS = OUTPUT_ROOT / 'embeddings'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
for path in [OUTPUT_ROOT, REPORTS, STATE, PAYLOADS, EMBEDDINGS]:
    path.mkdir(parents=True, exist_ok=True)

contract = {
    'run_tag': RUN_TAG,
    'partition_mode': PARTITION_MODE,
    'selected_strategies': SELECTED_STRATEGIES,
    'selected_sources': SELECTED_SOURCES,
    'active_strategies': ACTIVE_STRATEGIES,
    'active_sources': ACTIVE_SOURCES,
}
print('INPUT_ROOT =', INPUT_ROOT)
print('CORPUS_DIR =', CORPUS_DIR)
print('SCRIPTS_DIR =', SCRIPTS_DIR)
print('SOURCE_REGISTRY =', SOURCE_REGISTRY, 'exists =', SOURCE_REGISTRY.exists())
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print(json.dumps(contract, ensure_ascii=False, indent=2))

In [ ]:
requirements_file = SCRIPTS_DIR / 'backend' / 'requirements-kaggle.txt'
if requirements_file.exists():
    %pip install -q -r {requirements_file}
else:
    print('WARNING: requirements-kaggle.txt not found at', requirements_file)

In [ ]:
import subprocess, sys

required_inputs = [CORPUS_DIR / 'corpus', SOURCE_REGISTRY]
for path in required_inputs:
    print('input exists =', path.exists(), 'path =', path)
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError('Missing required corpus dataset paths: ' + ', '.join(missing_inputs))

expected_scripts = [
    SCRIPTS_DIR / 'scripts' / 'run_w3_ingest_07.py',
    SCRIPTS_DIR / 'scripts' / 'import_graph_payload.py',
    SCRIPTS_DIR / 'scripts' / 'import_embedding_artifacts.py',
]
for path in expected_scripts:
    print(path.name, 'exists =', path.exists())

cmd = [
    sys.executable, '-B', str(SCRIPTS_DIR / 'scripts' / 'run_w3_ingest_07.py'),
    '--mode', 'plan', '--profile', 'local-kaggle',
    '--dataset-dir', str(CORPUS_DIR),
    '--chunks-dir', str(OUTPUT_ROOT / 'chunks'),
    '--entities-dir', str(OUTPUT_ROOT / 'entities'),
    '--reports-dir', str(REPORTS),
]
subprocess.run(cmd, cwd=SCRIPTS_DIR, check=True)

manifest_path = REPORTS / 'w3_ingest_07_command_manifest.json'
print('Manifest path =', manifest_path)
print('Expected output layout =', ['chunks/', 'entities/', 'payloads/', 'embeddings/', 'reports/', 'state/'])